In [1]:
import pandas as pd
import numpy as np
import json
import os

pd.set_option('display.max_columns', 100)

In [2]:
prediction_df = pd.read_csv('../reports/02_explainability/prediction_types.csv')

global_shap_importance = pd.read_csv(
    '../reports/02_explainability/global_shap_importance.csv'
)

local_tp = pd.read_csv('../reports/02_explainability/local_explanation_tp.csv')
local_fn = pd.read_csv('../reports/02_explainability/local_explanation_fn.csv')
local_fp = pd.read_csv('../reports/02_explainability/local_explanation_fp.csv')
local_tn = pd.read_csv('../reports/02_explainability/local_explanation_tn.csv')

with open('../models/lgbm_tuned_clean_threshold.json') as f:
    final_threshold = json.load(f)['threshold']

print(f'Prediction records: {prediction_df.shape}')
print(f'Global SHAP features: {global_shap_importance.shape}')
print(f'Final threshold: {final_threshold}')

Prediction records: (18343, 4)
Global SHAP features: (263, 2)
Final threshold: 0.5


In [3]:
local_explanations = {
    'TP': local_tp,
    'FN': local_fn,
    'FP': local_fp,
    'TN': local_tn
}

for name, df in local_explanations.items():
    print(f'{name}: {df.shape}')

TP: (263, 4)
FN: (263, 4)
FP: (263, 4)
TN: (263, 4)


In [4]:
case_indices = {
    'TP': 14271,
    'FN': 1405,
    'FP': 10840,
    'TN': 15497
}

In [5]:
def make_json_safe(value):
    if pd.isna(value):
        return None

    if isinstance(value, str):
        stripped = value.strip()

        if stripped == 'True':
            return True

        if stripped == 'False':
            return False

        try:
            return float(stripped)
        except ValueError:
            return value

    if isinstance(value, (np.bool_, bool)):
        return bool(value)

    if isinstance(value, (np.integer, int)):
        return int(value)

    if isinstance(value, (np.floating, float)):
        return float(value)

    return value

In [6]:
clinical_meaning_map = {
    'age': 'older age is associated with higher mortality risk',
    'ventilated_apache': 'mechanical ventilation indicates severe respiratory failure or critical illness',
    'd1_spo2_min': 'low minimum oxygen saturation indicates hypoxemia',
    'd1_spo2_max': 'low maximum oxygen saturation may indicate persistent oxygenation impairment',
    'gcs_motor_apache': 'low GCS motor score indicates impaired neurological response',
    'gcs_verbal_apache': 'low GCS verbal score indicates impaired neurological response',
    'gcs_eyes_apache': 'low GCS eye score indicates impaired neurological response',
    'd1_sysbp_min': 'low systolic blood pressure indicates hemodynamic instability',
    'd1_mbp_min': 'low mean blood pressure indicates hemodynamic instability',
    'd1_heartrate_min': 'extreme or very low heart rate may indicate severe instability or data quality issue',
    'd1_heartrate_max': 'abnormal maximum heart rate may indicate physiological stress',
    'd1_resprate_min': 'extreme or very low respiratory rate may indicate respiratory instability or data quality issue',
    'd1_resprate_max': 'high respiratory rate may indicate respiratory distress',
    'd1_bun_max': 'elevated BUN may indicate renal dysfunction or metabolic stress',
    'd1_bun_min': 'BUN reflects renal function and metabolic status',
    'd1_hco3_min': 'low bicarbonate may indicate metabolic acidosis',
    'd1_hco3_max': 'low bicarbonate range may indicate metabolic acidosis',
    'd1_platelets_min': 'low platelet count may indicate severe illness or coagulation abnormalities',
    'd1_wbc_min': 'white blood cell count reflects inflammatory or immune status',
    'pre_icu_los_days': 'time between hospital admission and ICU admission; negative values require caution',
    'icu_id': 'ICU unit identifier; may reflect unit-level patterns rather than patient-level clinical status',
    'apache_3j_diagnosis': 'diagnosis category contributes baseline clinical severity information'
}

In [7]:
def get_caution_flags(feature, value):
    flags = []

    if feature == 'icu_id':
        flags.append('Non-clinical unit/location identifier; interpret cautiously.')

    if feature == 'pre_icu_los_days' and value is not None and value < 0:
        flags.append('Negative pre-ICU length of stay; may reflect timing or data quality issue.')

    zero_vital_features = {
        'd1_heartrate_min',
        'd1_resprate_min',
        'h1_resprate_min'
    }

    if feature in zero_vital_features and value == 0:
        flags.append('Zero-valued vital sign; may reflect extreme clinical event or recording artifact.')

    return flags

In [8]:
def build_evidence_packet(local_df, prediction_row, patient_label, test_row_index, top_n=8):
    def format_records(df, direction):
        records = []

        for _, row in df.iterrows():
            feature = row['feature']
            value = make_json_safe(row['value'])
            shap_value = float(row['shap_value'])

            records.append({
                'feature': feature,
                'value': value,
                'shap_value': shap_value,
                'direction': direction,
                'clinical_meaning': clinical_meaning_map.get(
                    feature,
                    'No predefined clinical interpretation available.'
                ),
                'caution_flags': get_caution_flags(feature, value)
            })

        return records

    risk_increasing_df = (
        local_df
        .sort_values('shap_value', ascending=False)
        .head(top_n)
        [['feature', 'value', 'shap_value']]
    )

    risk_decreasing_df = (
        local_df
        .sort_values('shap_value', ascending=True)
        .head(top_n)
        [['feature', 'value', 'shap_value']]
    )

    evidence_packet = {
        'patient_label': patient_label,
        'test_row_index': int(test_row_index),
        'prediction': {
            'y_true': int(prediction_row['y_true']),
            'y_pred': int(prediction_row['y_pred']),
            'y_proba': float(prediction_row['y_proba']),
            'prediction_type': prediction_row['prediction_type']
        },
        'risk_increasing_evidence': format_records(
            risk_increasing_df,
            direction='risk_increasing'
        ),
        'risk_decreasing_evidence': format_records(
            risk_decreasing_df,
            direction='risk_decreasing'
        )
    }

    return evidence_packet

In [9]:
evidence_packets = {
    case_type: build_evidence_packet(
        local_df=local_explanations[case_type],
        prediction_row=prediction_df.iloc[case_indices[case_type]],
        patient_label=f'{case_type}_case',
        test_row_index=case_indices[case_type],
        top_n=8
    )
    for case_type in ['TP', 'FN', 'FP', 'TN']
}

evidence_packets.keys()

dict_keys(['TP', 'FN', 'FP', 'TN'])

In [10]:
print(json.dumps(evidence_packets['TP'], indent=2))

{
  "patient_label": "TP_case",
  "test_row_index": 14271,
  "prediction": {
    "y_true": 1,
    "y_pred": 1,
    "y_proba": 0.999529083549892,
    "prediction_type": "TP"
  },
  "risk_increasing_evidence": [
    {
      "feature": "d1_heartrate_min",
      "value": 0.0,
      "shap_value": 2.0046439410543497,
      "direction": "risk_increasing",
      "clinical_meaning": "extreme or very low heart rate may indicate severe instability or data quality issue",
      "caution_flags": [
        "Zero-valued vital sign; may reflect extreme clinical event or recording artifact."
      ]
    },
    {
      "feature": "d1_spo2_min",
      "value": 31.0,
      "shap_value": 0.9487559313200912,
      "direction": "risk_increasing",
      "clinical_meaning": "low minimum oxygen saturation indicates hypoxemia",
      "caution_flags": []
    },
    {
      "feature": "ventilated_apache",
      "value": 1.0,
      "shap_value": 0.7250477735108737,
      "direction": "risk_increasing",
      "clini

In [11]:
os.makedirs('../reports/03_evidence', exist_ok=True)

with open('../reports/03_evidence/evidence_packets.json', 'w') as f:
    json.dump(evidence_packets, f, indent=2)

for case_type, packet in evidence_packets.items():
    with open(f'../reports/03_evidence/evidence_packet_{case_type.lower()}.json', 'w') as f:
        json.dump(packet, f, indent=2)

print('Evidence packets saved.')

Evidence packets saved.


## Evidence Construction Summary

This notebook converts saved SHAP outputs into structured evidence packets for selected TP, FN, FP, and TN cases. Each packet includes the model prediction, test row index, risk-increasing evidence, risk-decreasing evidence, clinical meaning for known features, and caution flags for variables requiring careful interpretation.

The purpose of this layer is to transform raw SHAP values into a structured clinical evidence format that can later be used by the LLM reasoning layer to generate patient-level explanations.